## ⚠️ Notebook status (updated dataset)

These notebooks were originally created as experiments (02–14). Their historical runs used **deprecated / non-integral data**.

As of **May 24, 2026**, the integral dataset is:
`dataset/raw/handover_dataset.csv`

Important dataset property:
- `optimal_cell_idx_in_k` is **constant 0** in the integral dataset because neighbor lists are score-sorted with the optimal cell at index 0.
- Any pipeline that trains directly on `optimal_cell_idx_in_k` without **neighbor-axis shuffling** will learn order leakage and produce misleading accuracy.

Recommended usage now:
- Treat notebooks 02–14 as **robustness / stability / scalability** harnesses.
- For a leakage-safe pointer target, reuse `src/preprocess.py` (`dataset/processed/*.npz`) or `src/production/temporal_deepset_data.py`.

See production docs:
- `docs/production_constraints.md`
- `docs/robustness_scalability_plan.md`
- `docs/explainability_and_finetuning.md`


In [35]:
# --- Multi-horizon window controls (NEW DATASET) ---
# Each row in the dataset corresponds to one measurement interval.
MEASUREMENT_INTERVAL_MS = 50

# History window length (timesteps). Example: 25 → 1.25 s history @ 50 ms.
WIN_T = 200

# Multi-horizon label generation. Example: 5 → predict up to 250 ms ahead.
FUTURE_H = 5

# Optional lead time before horizon starts (in timesteps).
LEAD_L = 0

# For notebooks that are NOT multi-output, choose which horizon to train on (1..FUTURE_H).
TARGET_H_IDX = 1

# Label source:
# - "optimal": train against oracle best cell (optimal_cell_id)
# - "target" : train against executed target (target_cell_id)
LABEL_MODE = "optimal"

# Feature toggles for cache builder:
# - include_scores=True adds nb_scores to per-cell features
# - include_global=True adds [speed, cos(dir), sin(dir), cell_load] to each cell feature vector
INCLUDE_SCORES = True
INCLUDE_GLOBAL = False

# Set True to force rebuilding dataset/mh_cache for new window/horizon settings
FORCE_REBUILD = False

# Use the leakage-safe multi-horizon cache loader.
USE_MH_CACHE = True

print(
    f"[window] dt={MEASUREMENT_INTERVAL_MS}ms  "
    f"T={WIN_T} ({WIN_T*MEASUREMENT_INTERVAL_MS}ms)  "
    f"H={FUTURE_H} ({FUTURE_H*MEASUREMENT_INTERVAL_MS}ms)  "
    f"lead={LEAD_L}  target_h={TARGET_H_IDX}"
)


[window] dt=50ms  T=200 (10000ms)  H=5 (250ms)  lead=0  target_h=1


In [36]:
# ─── SECTION 4: UNIFIED DATA PIPELINE (Multi-Horizon & Strictly Balanced) ───
# Provides X, M, y (multi-horizon), r (regression), and y_bin (binary HO).

SEED = 42
from pathlib import Path
import logging
try: _r = _ROOT
except NameError: _r = Path("../../").resolve()
try: log.info
except NameError: log = logging.getLogger("dummy"); log.setLevel(logging.INFO)
import re
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

def parse_nb_array(s, max_len=10, fill=0.0):
    if pd.isna(s): return [fill] * max_len
    cleaned = re.sub(r'[\[\]]', '', str(s)).strip()
    parts = re.split(r'[,;]', cleaned)
    vals = []
    for p in parts[:max_len]:
        p = p.strip()
        if p == '' or p.lower() in ('nan', 'none'): vals.append(fill)
        else:
            try: vals.append(float(p))
            except: vals.append(fill)
    vals += [fill] * (max_len - len(vals))
    return vals

def parse_nb_ids(s, max_len=10):
    if pd.isna(s): return [0] * max_len
    cleaned = re.sub(r'[\[\]]', '', str(s)).strip()
    parts = re.split(r'[,;]', cleaned)
    ids = []
    for p in parts[:max_len]:
        p = p.strip()
        try: ids.append(int(float(p)))
        except: ids.append(0)
    ids += [0] * (max_len - len(ids))
    return ids

def load_and_create_mh_datasets(root_dir, win_t=25, future_h=5, lead_l=0, k_cells=10):
    raw_path = root_dir / "dataset" / "raw" / "handover_dataset.csv"
    log.info(f"Loading raw data from {raw_path}...")
    df = pd.read_csv(raw_path, low_memory=False)
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed")

    df["nb_ids"]   = df["nb_cell_ids"].apply(parse_nb_ids)
    df["nb_rsrps"] = df["nb_rsrps"].apply(parse_nb_array)
    df["nb_sinrs"] = df["nb_sinrs"].apply(parse_nb_array)
    df["nb_loads"] = df["nb_loads"].apply(parse_nb_array)

    df.sort_values(["ue_id", "timestamp"], inplace=True)

    all_X, all_M, all_y, all_r, all_y_bin, groups = [], [], [], [], [], []
    rng_shuf = np.random.default_rng(SEED)

    log.info(f"Building MH sequences (T={win_t}, H={future_h}, L={lead_l})...")
    for ue_id, grp in df.groupby("ue_id", sort=False):
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        n_rows = len(grp)
        if n_rows < win_t + lead_l + future_h: continue

        cell_feat = np.zeros((n_rows, k_cells, 4), dtype=np.float32)
        for i in range(n_rows):
            rs, sn, ld = grp.at[i, "nb_rsrps"], grp.at[i, "nb_sinrs"], grp.at[i, "nb_loads"]
            for k in range(k_cells):
                cell_feat[i, k, 0] = rs[k]
                cell_feat[i, k, 1] = sn[k]
                cell_feat[i, k, 2] = ld[k]
                cell_feat[i, k, 3] = 0.0

        opt_ids    = grp["optimal_cell_id"].values
        ue_nb_ids  = grp["nb_ids"].values
        ue_srv_ids = grp["serving_cell_id"].values
        opt_rsrp   = grp["optimal_cell_rsrp"].values

        for t in range(win_t, n_rows - (lead_l + future_h) + 1):
            X_w = cell_feat[t-win_t : t]
            p = rng_shuf.permutation(k_cells)
            X_w_shuf = X_w[:, p, :].transpose(1, 0, 2)
            M_w = (X_w_shuf[:, -1, 0] != 0.0).astype(np.float32)

            yh    = np.zeros((future_h,), dtype=np.int32)
            rh    = np.zeros((future_h,), dtype=np.float32)
            y_bin = np.zeros((future_h,), dtype=np.int32)

            current_nb_ids = ue_nb_ids[t-1]
            current_srv    = int(ue_srv_ids[t-1])

            for step in range(future_h):
                f_idx     = t + lead_l + step
                chosen_id = int(opt_ids[f_idx])
                rh[step]  = float(opt_rsrp[f_idx])
                y_bin[step] = 1 if chosen_id != current_srv else 0
                try:
                    orig      = current_nb_ids.index(chosen_id)
                    yh[step]  = int(np.where(p == orig)[0][0])
                except ValueError:
                    srv = int(ue_srv_ids[f_idx])
                    try:
                        orig     = current_nb_ids.index(srv)
                        yh[step] = int(np.where(p == orig)[0][0])
                    except ValueError:
                        yh[step] = 0

            all_X.append(X_w_shuf)
            all_M.append(M_w)
            all_y.append(yh)
            all_r.append(rh)
            all_y_bin.append(y_bin)
            groups.append(ue_id)

    X     = np.array(all_X)
    M     = np.array(all_M)
    y     = np.array(all_y)
    r     = np.array(all_r, dtype=np.float32)
    y_bin = np.array(all_y_bin)
    groups= np.array(groups)

    # UE-level split (no temporal leakage)
    ue_list = np.unique(groups)
    np.random.default_rng(SEED).shuffle(ue_list)
    n_te, n_va = int(len(ue_list)*0.15), int(len(ue_list)*0.15)
    ue_te, ue_va = set(ue_list[:n_te]), set(ue_list[n_te:n_te+n_va])

    idx_tr = np.where([u not in ue_te and u not in ue_va for u in groups])[0]
    idx_va = np.where([u in ue_va for u in groups])[0]
    idx_te = np.where([u in ue_te for u in groups])[0]

    # Fit scalers on TRAIN only
    scaler_x = StandardScaler()
    scaler_x.fit(X[idx_tr][M[idx_tr]==1].reshape(-1, 4))
    X_n = X.copy()
    for i in range(len(X_n)):
        vk = np.where(M[i] == 1.0)[0]
        if len(vk) > 0:
            X_n[i, vk] = scaler_x.transform(X[i, vk].reshape(-1, 4)).reshape(-1, win_t, 4)

    scaler_r = StandardScaler()
    scaler_r.fit(r[idx_tr].reshape(-1, 1))
    r_n = scaler_r.transform(r.reshape(-1, 1)).reshape(-1, future_h)

    return {
        "train": {"X": X_n[idx_tr], "M": M[idx_tr], "y": y[idx_tr], "r": r_n[idx_tr], "y_bin": y_bin[idx_tr]},
        "val":   {"X": X_n[idx_va], "M": M[idx_va], "y": y[idx_va], "r": r_n[idx_va], "y_bin": y_bin[idx_va]},
        "test":  {"X": X_n[idx_te], "M": M[idx_te], "y": y[idx_te], "r": r_n[idx_te], "y_bin": y_bin[idx_te]},
        "scalers": {"x": scaler_x, "r": scaler_r}
    }

# Execute pipeline
try:    _wt = WIN_T
except: _wt = 25
try:    _fh = FUTURE_H
except: _fh = 5
try:    _ll = LEAD_L
except: _ll = 0

_data = load_and_create_mh_datasets(_r, win_t=_wt, future_h=_fh, lead_l=_ll, k_cells=10)

# Map to canonical multi-horizon variables
X_tr, M_tr, y_tr_mh, r_tr, y_bin_tr_mh = _data["train"]["X"], _data["train"]["M"], _data["train"]["y"], _data["train"]["r"], _data["train"]["y_bin"]
X_va, M_va, y_va_mh, r_va, y_bin_va_mh = _data["val"]["X"],   _data["val"]["M"],   _data["val"]["y"],   _data["val"]["r"],   _data["val"]["y_bin"]
X_te, M_te, y_te_mh, r_te, y_bin_te_mh = _data["test"]["X"],  _data["test"]["M"],  _data["test"]["y"],  _data["test"]["r"],  _data["test"]["y_bin"]
scaler   = _data["scalers"]["x"]
scaler_r = _data["scalers"]["r"]

# Multi-Horizon: preserve all H steps
y_tr = y_tr_mh.astype("int32")
y_va = y_va_mh.astype("int32")
y_te = y_te_mh.astype("int32")
y_bin_tr = y_bin_tr_mh.astype(np.float32)
y_bin_va = y_bin_va_mh.astype(np.float32)
y_bin_te = y_bin_te_mh.astype(np.float32)

try:
    if "N_FEATS" in HP: HP["N_FEATS"] = 3  # nb_score removed
except NameError:
    pass

log.info("MH Dataset ready: X_tr=%s  y_tr=%s  r_tr=%s", X_tr.shape, y_tr.shape, r_tr.shape)


12:41:36 │ INFO     │ Loading raw data from /home/wassimmchichi/Downloads/Handover_projects/dataset/raw/handover_dataset.csv...
12:41:42 │ INFO     │ Building MH sequences (T=200, H=5, L=0)...
12:42:02 │ INFO     │ MH Dataset ready: X_tr=(20370, 10, 200, 4)  y_tr=(20370, 5)  r_tr=(20370, 5)


# 04 · Experiment 4 — 6G Predictive Optimal Cell Selection
## Objective: Break the 57% ceiling via Spatial-Temporal MTL

### Paradigm Shift: Mimicry → Prediction
```
LEGACY (Notebooks 01–03):              6G PREDICTIVE (This Notebook):
  Target = handover_label               Target = optimal_cell_idx_in_k
  (delayed by Hysteresis + TTT)         (ground-truth optimal, zero latency)

  Features = 4–6 per cell               Features = 12 per cell
  (only RF quality)                      (RF + Δ-trajectory + mobility vector)

  Window = 25 steps (5 s)               Window = 50 steps (10 s)
  (insufficient trajectory context)      (captures full UE movement intent)
```

### Feature Layout (F = 12 per cell per timestep)
```
Index  Name             Source             Why
─────  ───────────────  ─────────────────  ───────────────────────────────────
  0    nb_rsrp          nb_rsrps           Signal strength of candidate cell
  1    nb_sinr          nb_sinrs           Interference quality
  2    nb_load          nb_loads           Cell utilisation
  3    nb_score         nb_scores          Pre-computed optimality composite
  4    rsrp_delta       computed           Fading velocity  (Δrsrp per step)
  5    rsrp_margin      computed           Gap vs serving cell (A3 trigger)
  6    ue_speed         speed              UE velocity → trajectory intent
  7    ue_direction     direction          UE heading → which cells lie ahead
  8    ue_altitude      altitude           UE height → drone vs ground path
  9    ue_sinr          sinr               Serving-cell SINR at measurement
 10    ue_cqi           cqi                Channel quality feedback
 11    ue_cell_load     cell_load          Serving cell current load
```

### Multi-Task Heads
```
Shared Encoder (LSTM+SetTransformer)
  │
  ├─ Head A (Focal Loss, w=1.0) → argmax optimal_cell_idx  [classification]
  └─ Head B (MSE Loss,   w=0.7) → predict optimal_cell_rsrp at t+1  [regression]
```

### Paths (from notebooks/modeling/)
```
../../dataset/6g_cache/   ← input
../../models/             ← best_6g_predictive.keras
../../metrics/            ← plots, metadata, CSV log
../../tb_logs/6g_predictive/  ← TensorBoard
../../mlflow/mlruns/      ← MLflow (EC2 via tunnel or env-var)
```

## Section 1 · Environment, Paths, GPU, MLflow

In [37]:
# ─── Section 1 · Environment ─────────────────────────────────────────────────
import os, sys, warnings, json, pickle, logging, datetime, gc, re, time
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing  import Tuple, List, Dict, Optional

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from sklearn.preprocessing      import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics            import (classification_report,
                                        confusion_matrix,
                                        top_k_accuracy_score,
                                        mean_absolute_error)
sns.set_theme(style="whitegrid", font_scale=1.05)

# ── Root anchor: notebooks/modeling/ → ../../ → project root ─────────────────
_ROOT = Path("../../").resolve()

PATHS = dict(
    raw_csv   = _ROOT / "dataset" / "raw"           / "handover_dataset.csv",
    cache_6g  = _ROOT / "dataset" / "6g_cache",
    models    = _ROOT / "models",
    metrics   = _ROOT / "metrics"/"6g_predictive",
    tb_logs   = _ROOT / "tb_logs" / "6g_predictive",
    mlruns    = _ROOT / "mlflow"  / "mlruns",
)
for p in (PATHS["cache_6g"], PATHS["models"],
          PATHS["metrics"], PATHS["tb_logs"], PATHS["mlruns"]):
    os.makedirs(str(p), exist_ok=True)

# ── Logging → stdout + metrics/6g_training.log ───────────────────────────────
_log_h = [logging.StreamHandler(sys.stdout),
          logging.FileHandler(str(PATHS["metrics"]/"6g_training.log"), mode="w")]
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s │ %(levelname)-8s │ %(message)s",
                    datefmt="%H:%M:%S", handlers=_log_h)
log = logging.getLogger("6g")
log.info("Root  : %s", _ROOT)
for k,v in PATHS.items(): log.info("  %-12s → %s", k, v)

# ── GPU ───────────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices("GPU")
log.info("GPUs: %d", len(gpus))
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
    log.info("  %s — memory_growth=True", g.name)
if not gpus:
    log.warning("No GPU detected — training on CPU.")

policy = mixed_precision.Policy("mixed_float16")
mixed_precision.set_global_policy(policy)
log.info("Mixed precision: compute=%s  vars=%s",
         policy.compute_dtype, policy.variable_dtype)

# ── MLflow → EC2 via SSH tunnel (or MLFLOW_TRACKING_URI env-var) ─────────────
_MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
try:
    import mlflow, mlflow.tensorflow, requests
    try:
        _r = requests.get(f"{_MLFLOW_URI}/health", timeout=3)
        MLFLOW_OK = _r.status_code == 200
    except Exception:
        MLFLOW_OK = False
    if MLFLOW_OK:
        mlflow.set_tracking_uri(_MLFLOW_URI)
        mlflow.set_experiment("handover_6G_Predictive")
        log.info("MLflow → %s", _MLFLOW_URI)
    else:
        log.warning("MLflow unreachable at %s (tunnel open?)", _MLFLOW_URI)
except ImportError:
    MLFLOW_OK = False
    log.warning("mlflow not installed — pip install mlflow")

SEED = 42
tf.random.set_seed(SEED); np.random.seed(SEED)
log.info("TF %s | NumPy %s", tf.__version__, np.__version__)

12:42:02 │ INFO     │ Root  : /home/wassimmchichi/Downloads/Handover_projects
12:42:02 │ INFO     │   raw_csv      → /home/wassimmchichi/Downloads/Handover_projects/dataset/raw/handover_dataset.csv
12:42:02 │ INFO     │   cache_6g     → /home/wassimmchichi/Downloads/Handover_projects/dataset/6g_cache
12:42:02 │ INFO     │   models       → /home/wassimmchichi/Downloads/Handover_projects/models
12:42:02 │ INFO     │   metrics      → /home/wassimmchichi/Downloads/Handover_projects/metrics/6g_predictive
12:42:02 │ INFO     │   tb_logs      → /home/wassimmchichi/Downloads/Handover_projects/tb_logs/6g_predictive
12:42:02 │ INFO     │   mlruns       → /home/wassimmchichi/Downloads/Handover_projects/mlflow/mlruns
12:42:02 │ INFO     │ GPUs: 1
12:42:02 │ INFO     │   /physical_device:GPU:0 — memory_growth=True
12:42:02 │ INFO     │ Mixed precision: compute=float16  vars=float32
12:42:02 │ INFO     │ MLflow → http://127.0.0.1:5000
12:42:02 │ INFO     │ TF 2.15.1 | NumPy 1.26.4


## Section 2 · Hyperparameters

In [38]:
# ─── Section 2 · Hyperparameters ─────────────────────────────────────────────

HP = dict(
    # ── Data geometry ─────────────────────────────────────────────────────────
    MAX_CELLS    = 10,
    OBS_STEPS    = 50,    # 10 s history @ 5 Hz — captures full trajectory
    N_FEATS      = 4,    # [nb_rsrp,nb_sinr,nb_load]  (nb_score removed — leakage)
                          #  speed,dir,alt,sinr,cqi,cell_load]
    LAT_STEPS    = 5,
    TGT_STEPS    = 5,

    # ── Loss weights ──────────────────────────────────────────────────────────
    LAMBDA_CLS   = 1.0,   # Focal Loss (Head A — classification)
    LAMBDA_REG   = 0.7,   # MSE Loss   (Head B — regression)
    FOCAL_GAMMA  = 2.0,
    FOCAL_ALPHA  = 0.25,

    # ── Architecture ──────────────────────────────────────────────────────────
    LSTM_UNITS   = 128,   # larger than Exp3 — more history to encode
    PHI_DIM      = 64,
    N_HEADS      = 4,
    MHA_KEY_DIM  = 16,
    FF_DIM       = 128,
    N_ST_BLOCKS  = 2,
    DROPOUT      = 0.20,

    # ── Training ──────────────────────────────────────────────────────────────
    BATCH_SIZE   = 64,    # 6 GB VRAM × (10,50,12) tensors
    EPOCHS       = 60,
    LR_INIT      = 5e-4,  # lower init — new spatial features need gentle start
    LR_WARMUP_EP = 5,     # 5-epoch linear warm-up for spatial feature stability
    LR_DECAY_EP  = 25,
)

ALL_LABELS = list(range(HP["MAX_CELLS"]))
F_NAMES = ["nb_rsrp", "nb_sinr", "nb_load"]  # nb_score removed — leakage

CKPT_PATH  = str(PATHS["models"] / "best_6g_predictive.keras")
FINAL_PATH = str(PATHS["models"] / "6g_predictive_final.keras")

log.info("HP loaded — OBS_STEPS=%d  N_FEATS=%d  LSTM=%d",
         HP["OBS_STEPS"], HP["N_FEATS"], HP["LSTM_UNITS"])
log.info("Loss: %.1f×Focal + %.1f×MSE  LR_INIT=%.0e  WARMUP=%dep",
         HP["LAMBDA_CLS"], HP["LAMBDA_REG"], HP["LR_INIT"], HP["LR_WARMUP_EP"])

# --- Override window controls (injected) ---
try:
    HP["OBS_STEPS"] = int(WIN_T)
    if "PRED_STEPS" in HP: HP["PRED_STEPS"] = int(FUTURE_H)
    if "TGT_STEPS"  in HP: HP["TGT_STEPS"]  = int(FUTURE_H)
    if "LAT_STEPS"  in HP: HP["LAT_STEPS"]  = int(LEAD_L)
except Exception as _e:
    print('HP override skipped:', _e)


12:42:02 │ INFO     │ HP loaded — OBS_STEPS=50  N_FEATS=4  LSTM=128
12:42:02 │ INFO     │ Loss: 1.0×Focal + 0.7×MSE  LR_INIT=5e-04  WARMUP=5ep


## Section 3 · Data Pipeline — String-to-Tensor Neighbour Parser

### Why vectorised parsing (not row-by-row)

The neighbour columns (`nb_rsrps`, `nb_sinrs`, `nb_loads`, `nb_scores`) store
8 floats as `"[-44.00;-65.92;…]"` strings. Parsing one row at a time inside
the window loop costs ~11 minutes for 80 UEs.

The vectorised strategy:
1. **Parse once** — convert all 47,920 rows of each list column to a
   `(47920, 10)` float32 numpy array in a single pass (~1 s total).
2. **Pre-assemble** `FEAT_ALL (47920, 10, 12)` — one global matrix of all
   features indexed by CSV row number.
3. **Slice per window** — `FEAT_ALL[obs_indices].transpose(1,0,2)` gives
   `(10, 50, 12)` in microseconds via numpy strided views.
4. **Fill Δ and margin in-place** — `np.diff` on the RSRP slice is
   vectorised across all 10 cells at once.

This reduces total build time from **~11 min → ~5 s**.

In [39]:
# ─── Section 3a · Vectorised neighbour list parser ───────────────────────────

_BRACKET = re.compile(r'[\[\]]')

def parse_list_col(series: pd.Series,
                   k: int = HP["MAX_CELLS"]) -> np.ndarray:
    """
    Convert a full column of '[v0;v1;…;vk-1]' strings to a (N, k) float32 array.

    Design choices:
    • Processes the entire column in one Python pass (not via .apply()).
    • NaN is used for missing / unparseable entries; replaced with 0.0 after
      the mask has been computed.
    • Bracket regex compiled once at module level.

    Args:
        series : pd.Series of raw strings, length N
        k      : output width (MAX_CELLS = 10)

    Returns:
        np.ndarray shape (N, k) dtype float32
    """
    N      = len(series)
    result = np.full((N, k), np.nan, dtype=np.float32)
    for i, raw in enumerate(series):
        s = _BRACKET.sub("", str(raw)).strip()
        for j, p in enumerate(s.split(";")[:k]):
            try:
                result[i, j] = float(p.strip())
            except ValueError:
                pass
    return result


def build_global_feature_matrix(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    """
    Pre-assemble FEAT_ALL (N_rows, MAX_CELLS, N_FEATS) once for the whole CSV.

    Feature layout:
      [0] nb_rsrp      — neighbour RSRP         (dBm)
      [1] nb_sinr      — neighbour SINR          (dB)
      [2] nb_load      — neighbour cell load     (0–1)
      [3] nb_score     — optimality composite    (0–1)
      [4] rsrp_delta   — Δrsrp step-to-step      (filled per window)
      [5] rsrp_margin  — rsrp_i − rsrp_serving   (filled per window)
      [6] ue_speed     — UE speed                (km/h)
      [7] ue_direction — UE heading              (degrees)
      [8] ue_altitude  — UE altitudef             (m)
      [9] ue_sinr      — serving-cell SINR        (dB)
     [10] ue_cqi       — CQI feedback             (1–15)
     [11] ue_cell_load — serving cell load        (0–1)

    Returns:
        FEAT_ALL : (N, MAX_CELLS, N_FEATS) float32
        MASK_ARR : (N, MAX_CELLS) float32  — 1=real cell, 0=padding
    """
    N  = len(df)
    MC = HP["MAX_CELLS"]
    F  = HP["N_FEATS"]

    log.info("Parsing list columns …")
    NB_RSRPS  = parse_list_col(df["nb_rsrps"])    # (N, 10)
    NB_SINRS  = parse_list_col(df["nb_sinrs"])
    NB_LOADS  = parse_list_col(df["nb_loads"])
    # NB_SCORES removed — encodes selection criterion (leakage)

    # Mask before NaN fill
    MASK = (~np.isnan(NB_RSRPS)).astype(np.float32)  # (N, 10)

    for arr in (NB_RSRPS, NB_SINRS, NB_LOADS):
        np.nan_to_num(arr, nan=0.0, copy=False)

    log.info("Assembling FEAT_ALL …")
    FEAT = np.zeros((N, MC, F), dtype=np.float32)

    # Neighbour RF features (columns 0-3)
    FEAT[:, :, 0] = NB_RSRPS
    FEAT[:, :, 1] = NB_SINRS
    FEAT[:, :, 2] = NB_LOADS
    # FEAT[:, :, 3] = NB_SCORES  removed — leakage
    # Columns 4-5 (delta, margin) filled per window — left as zero here

    # UE scalar features — broadcast across MAX_CELLS axis (all cells see same UE state)
    FEAT[:, :, 6]  = df["speed"].values.astype(np.float32)[:, None]
    FEAT[:, :, 7]  = df["direction"].values.astype(np.float32)[:, None]
    FEAT[:, :, 8]  = df["altitude"].values.astype(np.float32)[:, None]
    FEAT[:, :, 9]  = df["sinr"].values.astype(np.float32)[:, None]
    FEAT[:, :, 10] = df["cqi"].values.astype(np.float32)[:, None]
    FEAT[:, :, 11] = df["cell_load"].values.astype(np.float32)[:, None]

    log.info("FEAT_ALL: %s  %.2f GB", FEAT.shape, FEAT.nbytes / 1e9)
    return FEAT, MASK


log.info("Neighbour parser and feature-matrix builder defined.")

12:42:02 │ INFO     │ Neighbour parser and feature-matrix builder defined.


In [40]:
# ─── Section 3b · Window builder (vectorised, fast) ──────────────────────────

def build_ue_windows(
    row_indices: np.ndarray,
    FEAT_ALL:    np.ndarray,
    MASK_ARR:    np.ndarray,
    OPT_IDX:     np.ndarray,
    OPT_RSRP:    np.ndarray,
    obs:   int = HP["OBS_STEPS"],
    gap:   int = HP["LAT_STEPS"],
    tgt:   int = HP["TGT_STEPS"],
    mc:    int = HP["MAX_CELLS"],
    n_f:   int = HP["N_FEATS"],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Build sliding windows for one UE using pre-assembled global arrays.

    Strategy (vectorised):
      1. Slice FEAT_ALL[obs_indices] → (OBS, MC, F)
      2. Transpose → (MC, OBS, F) in one numpy op
      3. Fill F[4]=Δrsrp and F[5]=margin in-place (np.diff + broadcast)
      4. Read label/regression target from pre-computed 1-D arrays

    Labels:
      y (int)  = optimal_cell_idx_in_k at last target row
                 (ground-truth index; always 0–7 in this dataset)
      r (float) = mean optimal_cell_rsrp over 5 target rows

    Args:
        row_indices : sorted CSV row indices for this UE

    Returns:
        X : (N_win, MC, OBS, N_FEATS) float32
        M : (N_win, MC)               float32 mask
        y : (N_win,)                  int32   classification label
        r : (N_win,)                  float32 regression target (raw dBm)
    """
    n     = len(row_indices)
    n_win = n - obs - gap - tgt + 1

    X = np.zeros((n_win, mc, obs, n_f), dtype=np.float32)
    M = np.zeros((n_win, mc),           dtype=np.float32)
    y = np.zeros((n_win, tgt),          dtype=np.int32)
    r = np.zeros((n_win, tgt),          dtype=np.float32)

    for w in range(n_win):
        obs_idx = row_indices[w : w + obs]
        tgt_idx = row_indices[w + obs + gap : w + obs + gap + tgt]

        # Core slice + transpose (vectorised, no Python loop over timesteps)
        win = FEAT_ALL[obs_idx].transpose(1, 0, 2)   # (MC, OBS, F)
        X[w] = win

        # In-place trajectory features
        rsrp_seq  = win[:, :, 0]                                       # (MC, OBS)
        X[w, :, :, 4] = np.diff(rsrp_seq, axis=1,
                                 prepend=rsrp_seq[:, :1])              # Δ
        X[w, :, :, 5] = rsrp_seq - rsrp_seq[0:1, :]                   # margin

        # Mask = union (any real cell across the observation window)
        M[w] = MASK_ARR[obs_idx].max(axis=0)

        # Labels from pre-indexed arrays (O(1) access)
        y[w] = OPT_IDX[tgt_idx].astype(np.int32)
        r[w] = OPT_RSRP[tgt_idx].astype(np.float32)

    return X, M, y, r


def build_split(ue_list: List[str],
                ue_groups: Dict[str, np.ndarray],
                FEAT_ALL: np.ndarray,
                MASK_ARR: np.ndarray,
                OPT_IDX:  np.ndarray,
                OPT_RSRP: np.ndarray,
                name: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    Xs, Ms, ys, rs = [], [], [], []
    for uid in ue_list:
        X, M, y, r = build_ue_windows(
            ue_groups[uid], FEAT_ALL, MASK_ARR, OPT_IDX, OPT_RSRP)
        Xs.append(X); Ms.append(M); ys.append(y); rs.append(r)
    return (np.concatenate(Xs).astype(np.float32),
            np.concatenate(Ms).astype(np.float32),
            np.concatenate(ys),
            np.concatenate(rs))

log.info("Window builder defined.")

12:42:02 │ INFO     │ Window builder defined.


## Section 4 · tf.data Pipeline — Dual-Output, One-Hot Labels

In [41]:
# ─── Section 4 · Data pipeline ───────────────────────────────────────────────

AUTOTUNE = tf.data.AUTOTUNE

cw_vals      = compute_class_weight("balanced",
                                     classes=np.unique(y_tr), y=y_tr.flatten())
CLASS_WEIGHT = {int(c): float(w) for c, w in enumerate(cw_vals)}
log.info("Class weights: %s",
         {k: round(v,3) for k,v in CLASS_WEIGHT.items()})


def make_ds(X: np.ndarray, M: np.ndarray,
            y: np.ndarray, r: np.ndarray,
            sw: np.ndarray = None, 
            shuffle: bool = False) -> tf.data.Dataset:
    """
    Build a dual-output tf.data.Dataset.

    Inputs : {"cells": X, "mask": M}
    Outputs: {"cls_output": y_one_hot, "reg_output": r_scaled}
    Weights: {"cls_output": sw} (Only applied if sw is provided)
    """
    if y.ndim == 2:  # Multi-horizon: (N, H) -> (N, H, C)
        y_oh = tf.one_hot(y, depth=HP["MAX_CELLS"]).numpy().astype(np.float32)
    else:
        y_oh = tf.one_hot(y, depth=HP["MAX_CELLS"]).numpy().astype(np.float32)
    
    inputs  = {"cells": X, "mask": M}
    targets = {"cls_output": y_oh, "reg_output": r.astype(np.float32)}
    
    with tf.device('/CPU:0'):
        if sw is not None:
            # Map sample weights specifically to the classification output
            weights = {"cls_output": sw.astype(np.float32)}
            ds = tf.data.Dataset.from_tensor_slices((inputs, targets, weights))
        else:
            ds = tf.data.Dataset.from_tensor_slices((inputs, targets))
        
    if shuffle:
        ds = ds.shuffle(len(y), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(HP["BATCH_SIZE"], drop_remainder=False).prefetch(AUTOTUNE)


# Map the CLASS_WEIGHT dict to an array of weights for every training sample
sw_tr = np.mean([[CLASS_WEIGHT[int(val)] for val in row] for row in y_tr], axis=1).astype(np.float32)

# Pass the sample weights only to the training dataset
ds_tr = make_ds(X_tr, M_tr, y_tr, r_tr, sw=sw_tr, shuffle=True)
ds_va = make_ds(X_va, M_va, y_va, r_va)
ds_te = make_ds(X_te, M_te, y_te, r_te)

steps_per_epoch = len(ds_tr)
log.info("Batches → train:%d  val:%d  test:%d  steps/ep:%d",
         len(ds_tr), len(ds_va), len(ds_te), steps_per_epoch)

12:42:02 │ INFO     │ Class weights: {0: 0.877, 1: 0.996, 2: 1.006, 3: 0.991, 4: 1.029, 5: 1.005, 6: 1.029, 7: 1.025, 8: 1.012, 9: 1.052}
12:42:03 │ INFO     │ Batches → train:319  val:69  test:69  steps/ep:319


2026-05-30 12:42:03.161102: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 651840000 exceeds 10% of free system memory.


## Section 5 · Loss Functions — Focal + MSE

In [42]:
# ─── Section 5 · Losses ──────────────────────────────────────────────────────

def focal_loss(gamma: float = 2.0, alpha: float = 0.25):
    """
    Multi-class Focal Loss for dense one-hot targets.
    FL = −α · (1 − p_t)^γ · log(p_t)
    p_t = Σ y_true · y_pred (probability on the correct class).
    """
    def _loss(y_true, y_pred):
        y_pred  = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1-1e-7)
        y_true  = tf.cast(y_true, tf.float32)
        ce      = -y_true * tf.math.log(y_pred)
        p_t     = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        weight  = alpha * tf.pow(1.0 - p_t, gamma)
        return tf.reduce_sum(weight * ce, axis=-1)
    _loss.__name__ = f"focal_g{gamma}_a{alpha}"
    return _loss


FOCAL_LOSS = focal_loss(HP["FOCAL_GAMMA"], HP["FOCAL_ALPHA"])
MSE_LOSS   = tf.keras.losses.MeanSquaredError()

log.info("Focal: γ=%.1f  α=%.2f  |  MSE on standardised RSRP",
         HP["FOCAL_GAMMA"], HP["FOCAL_ALPHA"])

12:42:03 │ INFO     │ Focal: γ=2.0  α=0.25  |  MSE on standardised RSRP


## Section 6 · Custom Layers — Masked MHA & Set Transformer Block

In [43]:
# ─── Section 6 · Custom layers ───────────────────────────────────────────────

class MaskedMultiHeadAttention(keras.layers.Layer):
    """
    Multi-Head Self-Attention over MAX_CELLS with boolean key masking.

    key_mask (B, 1, 1, C):  True = attend, False = ignore (padding).
    dtype = float32 for numerical stability with mixed_float16 policy.
    """
    def __init__(self, num_heads, key_dim, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=key_dim,
            dropout=dropout, dtype="float32")
        self._cfg = {"num_heads":num_heads,"key_dim":key_dim,"dropout":dropout}

    def call(self, x, mask, training=False):
        key_mask = tf.cast(mask, tf.bool)[:, tf.newaxis, tf.newaxis, :]
        return self.mha(query=x, value=x, key=x,
                        attention_mask=key_mask, training=training)

    def get_config(self):
        return {**super().get_config(), **self._cfg}


class SetTransformerBlock(keras.layers.Layer):
    """
    Pre-LN Set Transformer block applied over the MAX_CELLS axis.

    Pre-LayerNorm (Wang 2019) + residual connections + masked MHA + GELU FFN.
    Padding cells are explicitly zeroed after each block — prevents their
    zero-initialised representations from accumulating residual signal.
    """
    def __init__(self, embed_dim, num_heads, key_dim, ff_dim,
                 dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self._cfg = dict(embed_dim=embed_dim,num_heads=num_heads,
                         key_dim=key_dim,ff_dim=ff_dim,dropout=dropout)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6, dtype="float32")
        self.norm2 = layers.LayerNormalization(epsilon=1e-6, dtype="float32")
        self.mha   = MaskedMultiHeadAttention(num_heads, key_dim, dropout)
        self.ff1   = layers.Dense(ff_dim,    activation="gelu")
        self.ff2   = layers.Dense(embed_dim, activation=None)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)

    def call(self, x, mask, training=False):
        x = x + self.drop1(self.mha(self.norm1(x), mask, training), training)
        x = x + self.drop2(self.ff2(self.ff1(self.norm2(x))), training)
        return x * tf.cast(mask[:, :, tf.newaxis], x.dtype)

    def get_config(self):
        return {**super().get_config(), **self._cfg}

print("MaskedMultiHeadAttention  ✓")
print("SetTransformerBlock       ✓")

MaskedMultiHeadAttention  ✓
SetTransformerBlock       ✓


## Section 7 · 6G Predictive MTL Model

### Input shape change: `(B, 10, 50, 12)` vs previous `(B, 10, 25, 4–6)`

The 50-step window provides 10 seconds of history — enough to observe a
full approach trajectory (pedestrian at 5 km/h covers ~14 m; highway UE
at 60 km/h covers ~167 m in 10 s). The mobility vector `(speed, direction,
altitude)` lets the model learn that a UE heading at 45° toward Cell 3
will need that cell in ~5 s even before the RSRP rises enough to trigger A3.

In [44]:
# ─── Section 7 · 6G Predictive MTL Model ─────────────────────────────────────

def build_6g_predictive_model(hp: dict) -> keras.Model:
    C = hp["MAX_CELLS"]; W = hp["OBS_STEPS"]
    F = hp["N_FEATS"];   D = hp["PHI_DIM"]

    # ── Inputs ────────────────────────────────────────────────────────────────
    inp_cells = keras.Input((C, W, F), name="cells",  dtype="float32")
    inp_mask  = keras.Input((C,),      name="mask",   dtype="float32")

    # ── Shared Stage 1: Temporal Encoder ─────────────────────────────────────
    # Larger LSTM (128) than Exp3 — 50-step sequences carry more temporal state
    trend = layers.TimeDistributed(
        layers.LSTM(hp["LSTM_UNITS"], return_sequences=False),
        name="td_lstm")(inp_cells)                              # (B, C, 128)

    # ── Shared Stage 2: Φ projection ─────────────────────────────────────────
    phi = layers.TimeDistributed(
        layers.Dense(D, activation="relu"), name="phi")(trend)
    phi = layers.TimeDistributed(
        layers.Dropout(hp["DROPOUT"]), name="phi_drop")(phi)   # (B, C, 64)

    # ── Shared Stage 3: Set Transformer — pairwise attention ─────────────────
    x = phi
    for i in range(hp["N_ST_BLOCKS"]):
        x = SetTransformerBlock(
            embed_dim = D,
            num_heads = hp["N_HEADS"],
            key_dim   = hp["MHA_KEY_DIM"],
            ff_dim    = hp["FF_DIM"],
            dropout   = hp["DROPOUT"],
            name      = f"st_{i}",
        )(x, inp_mask)                                          # (B, C, 64)

    # ── Head A: Classification — optimal cell selection ───────────────────────
    rho = layers.TimeDistributed(
        layers.Dense(D, activation="relu"), name="rho_cls")(x)
    rho = layers.TimeDistributed(
        layers.Dropout(hp["DROPOUT"]), name="rho_cls_drop")(rho)
        
    H = hp.get("TGT_STEPS", 5)
    logits = layers.TimeDistributed(
        layers.Dense(H, use_bias=True), name="scorer")(rho)
    logits = layers.Permute((2, 1), name="logits_permuted")(logits)  # (B, H, C)
    
    cls_out = layers.Softmax(axis=-1, dtype="float32", name="cls_output")(
        layers.Add(name="pad_mask")([logits, layers.Reshape((1, C))((1.0 - inp_mask) * (-1e9))])
    )                                                            # (B, H, C)

    # ── Head B: Regression — predict target-cell RSRP ────────────────────────
    # Masked mean-pool over real cells → environment summary
    mask_exp  = layers.Reshape((C, 1), name="mask_exp")(inp_mask)
    x_sum     = tf.reduce_sum(x * tf.cast(mask_exp, x.dtype), axis=1)
    x_cnt     = tf.maximum(tf.reduce_sum(mask_exp, axis=1), 1e-8)
    z_pool    = x_sum / tf.cast(x_cnt, x_sum.dtype)            # (B, D)
    reg       = layers.Dense(32, activation="relu",  name="reg_fc")(z_pool)
    reg       = layers.Dropout(hp["DROPOUT"],        name="reg_drop")(reg)
    H = hp.get("TGT_STEPS", 5)
    reg_out   = layers.Dense(H, activation=None,
                              dtype="float32", name="reg_output")(reg)  # (B, H)

    model = keras.Model(
        inputs  = [inp_cells, inp_mask],
        outputs = {"cls_output": cls_out, "reg_output": reg_out},
        name    = "6G_Predictive_MTL",
    )
    return model


model = build_6g_predictive_model(HP)
model.summary(line_length=90, expand_nested=False)
log.info("Parameters: %d  |  Input: (B,%d,%d,%d)",
         model.count_params(), HP["MAX_CELLS"], HP["OBS_STEPS"], HP["N_FEATS"])

Model: "6G_Predictive_MTL"
__________________________________________________________________________________________
 Layer (type)              Output Shape               Param   Connected to                
                                                       #                                  
 cells (InputLayer)        [(None, 10, 200, 4)]       0       []                          
                                                                                          
 td_lstm (TimeDistributed  (None, 10, 128)            68096   ['cells[0][0]']             
 )                                                                                        
                                                                                          
 phi (TimeDistributed)     (None, 10, 64)             8256    ['td_lstm[0][0]']           
                                                                                          
 phi_drop (TimeDistribute  (None, 10, 64)             0       [

## Section 8 · Compile — Weighted MTL Loss + LR Schedule

In [45]:
# ─── Section 8 · LR schedule + compile ───────────────────────────────────────

class WarmUpCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    """Step-based linear warm-up → cosine decay → flat at lr_min."""
    def __init__(self, lr_max, lr_min, warmup_steps, decay_steps):
        super().__init__()
        self.lr_max=float(lr_max); self.lr_min=float(lr_min)
        self.warmup_steps=float(warmup_steps)
        self.decay_steps=float(decay_steps)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        warmup = self.lr_max * step / tf.maximum(self.warmup_steps, 1.0)
        cosine = self.lr_min + 0.5*(self.lr_max-self.lr_min)*(
            1.0+tf.cos(np.pi*tf.minimum(
                step-self.warmup_steps,self.decay_steps)/self.decay_steps))
        return tf.where(step < self.warmup_steps, warmup, cosine)

    def get_config(self):
        return {"lr_max":self.lr_max,"lr_min":self.lr_min,
                "warmup_steps":self.warmup_steps,"decay_steps":self.decay_steps}


warmup_steps = HP["LR_WARMUP_EP"] * steps_per_epoch
decay_steps  = HP["LR_DECAY_EP"]  * steps_per_epoch
lr_sched     = WarmUpCosineDecay(HP["LR_INIT"], HP["LR_INIT"]*0.01,
                                  warmup_steps, decay_steps)

model.compile(
    optimizer    = keras.optimizers.Adam(learning_rate=lr_sched),
    loss         = {"cls_output": FOCAL_LOSS, "reg_output": MSE_LOSS},
    loss_weights = {"cls_output": HP["LAMBDA_CLS"],
                    "reg_output": HP["LAMBDA_REG"]},
    metrics      = {
        "cls_output": [
            keras.metrics.CategoricalAccuracy(name="top1_acc"),
            keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
        ],
        "reg_output": [
            keras.metrics.MeanAbsoluteError(name="rsrp_mae"),
        ],
    },
)
log.info("Compiled. Loss=%.1f×Focal+%.1f×MSE  LR=%.0e  warmup=%d steps",
         HP["LAMBDA_CLS"],HP["LAMBDA_REG"],HP["LR_INIT"],warmup_steps)

# LR curve → metrics/
ep_lr=[float(lr_sched(e*steps_per_epoch)) for e in range(HP["EPOCHS"])]
fig,ax=plt.subplots(figsize=(8,3))
ax.plot(ep_lr,lw=2,color="#2196F3")
ax.axvline(HP["LR_WARMUP_EP"],color="orange",ls="--",lw=1.2,
           label=f"Warm-up end (ep{HP['LR_WARMUP_EP']})")
ax.axvline(HP["LR_WARMUP_EP"]+HP["LR_DECAY_EP"],color="red",ls="--",lw=1.2)
ax.set(xlabel="Epoch",ylabel="LR",title="WarmUpCosineDecay (LR_INIT=5e-4, WARMUP=5ep)")
ax.legend(fontsize=9); ax.grid(alpha=0.4)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f"{x:.1e}"))
plt.tight_layout()
plt.savefig(str(PATHS["metrics"]/"6g_lr_schedule.png"),dpi=150,bbox_inches="tight")
plt.close()

12:42:04 │ INFO     │ Compiled. Loss=1.0×Focal+0.7×MSE  LR=5e-04  warmup=1595 steps


In [46]:
# ─── Section 8.5 · Hyperparameter Tuning with Optuna ─────────────────────────
import optuna
import copy
from optuna.integration import TFKerasPruningCallback

def objective(trial):
    # 1. Suggest hyperparameters
    hp_trial = copy.deepcopy(HP)
    hp_trial['LSTM_UNITS'] = trial.suggest_categorical('LSTM_UNITS', [64, 128, 256])
    hp_trial['PHI_DIM'] = trial.suggest_categorical('PHI_DIM', [32, 64, 128])
    hp_trial['N_HEADS'] = trial.suggest_categorical('N_HEADS', [2, 4, 8])
    hp_trial['DROPOUT'] = trial.suggest_float('DROPOUT', 0.1, 0.4)
    hp_trial['LR_INIT'] = trial.suggest_float('LR_INIT', 1e-4, 5e-3, log=True)
    
    # 2. Build model
    model_trial = build_6g_predictive_model(hp_trial)
    
    # 3. Compile model
    warmup_steps = hp_trial['LR_WARMUP_EP'] * steps_per_epoch
    decay_steps  = hp_trial['LR_DECAY_EP']  * steps_per_epoch
    lr_sched_trial = WarmUpCosineDecay(hp_trial['LR_INIT'], hp_trial['LR_INIT']*0.01,
                                      warmup_steps, decay_steps)
                                      
    model_trial.compile(
        optimizer    = keras.optimizers.Adam(learning_rate=lr_sched_trial),
        loss         = {'cls_output': FOCAL_LOSS, 'reg_output': MSE_LOSS},
        loss_weights = {'cls_output': hp_trial['LAMBDA_CLS'],
                        'reg_output': hp_trial['LAMBDA_REG']},
        metrics      = {'cls_output': [keras.metrics.CategoricalAccuracy(name='top1_acc')]}
    )
    
    # 4. Train for a short period with pruning
    history = model_trial.fit(
        ds_tr,
        validation_data=ds_va,
        epochs=10, # Tune for 10 epochs
        callbacks=[TFKerasPruningCallback(trial, 'val_cls_output_top1_acc')],
        verbose=0
    )
    
    return max(history.history['val_cls_output_top1_acc'])

# Set RUN_OPTUNA to True to run hyperparameter tuning
RUN_OPTUNA = True

if RUN_OPTUNA:
    log.info('Starting Optuna hyperparameter tuning...')
    study = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=10)
    
    log.info('Best trial: %s', study.best_trial.value)
    log.info('Best params: %s', study.best_params)
    
    # Update HP with best params
    for k, v in study.best_params.items():
        HP[k] = v
        
    log.info('Updated HP with Optuna best parameters.')
    
    # Rebuild and recompile final model with optimized parameters
    model = build_6g_predictive_model(HP)
    
    warmup_steps = HP['LR_WARMUP_EP'] * steps_per_epoch
    decay_steps  = HP['LR_DECAY_EP']  * steps_per_epoch
    lr_sched_final = WarmUpCosineDecay(HP['LR_INIT'], HP['LR_INIT']*0.01,
                                      warmup_steps, decay_steps)
    model.compile(
        optimizer    = keras.optimizers.Adam(learning_rate=lr_sched_final),
        loss         = {'cls_output': FOCAL_LOSS, 'reg_output': MSE_LOSS},
        loss_weights = {'cls_output': HP['LAMBDA_CLS'],
                        'reg_output': HP['LAMBDA_REG']},
        metrics      = {
            'cls_output': [
                keras.metrics.CategoricalAccuracy(name='top1_acc'),
                keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc'),
            ],
            'reg_output': [
                keras.metrics.MeanAbsoluteError(name='rsrp_mae'),
            ],
        },
    )
    log.info('Rebuilt and recompiled final model with optimized HP.')


12:42:05 │ INFO     │ Starting Optuna hyperparameter tuning...


[I 2026-05-30 12:42:05,271] A new study created in memory with name: no-name-d6237b7e-35fc-4baf-94cf-f5fb796418e6
2026-05-30 12:42:06.025192: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 651840000 exceeds 10% of free system memory.
[I 2026-05-30 12:44:46,388] Trial 0 finished with value: 0.5159679055213928 and parameters: {'LSTM_UNITS': 256, 'PHI_DIM': 64, 'N_HEADS': 8, 'DROPOUT': 0.2330280751094844, 'LR_INIT': 0.0002492372208772064}. Best is trial 0 with value: 0.5159679055213928.
[I 2026-05-30 12:46:29,563] Trial 1 finished with value: 0.5184879899024963 and parameters: {'LSTM_UNITS': 128, 'PHI_DIM': 64, 'N_HEADS': 2, 'DROPOUT': 0.22488043011032868, 'LR_INIT': 0.0009974316591943697}. Best is trial 1 with value: 0.5184879899024963.
[I 2026-05-30 12:49:13,273] Trial 2 finished with value: 0.5204581618309021 and parameters: {'LSTM_UNITS': 256, 'PHI_DIM': 64, 'N_HEADS': 2, 'DROPOUT': 0.3781683370465745, 'LR_INIT': 0.001682861439081382}. Best is trial 2 with 

12:58:43 │ INFO     │ Best trial: 0.5204581618309021
12:58:43 │ INFO     │ Best params: {'LSTM_UNITS': 256, 'PHI_DIM': 64, 'N_HEADS': 2, 'DROPOUT': 0.3781683370465745, 'LR_INIT': 0.001682861439081382}
12:58:43 │ INFO     │ Updated HP with Optuna best parameters.
12:58:44 │ INFO     │ Rebuilt and recompiled final model with optimized HP.


## Section 9 · Callbacks & Training

In [53]:
# ─── Section 9 · Callbacks ───────────────────────────────────────────────────

MONITOR = "val_cls_output_top1_acc"

class MLflowEpochCB(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if MLFLOW_OK and logs:
            for k,v in logs.items(): mlflow.log_metric(k,float(v),step=epoch)

if MLFLOW_OK:
    mlflow.end_run()
    _run = mlflow.start_run(run_name="6G_predictive_")
    mlflow.log_params(HP)
    mlflow.log_param("loss_fn",
                     f"{HP['LAMBDA_CLS']}xFocal+{HP['LAMBDA_REG']}xMSE")
    mlflow.log_param("target", "optimal_cell_idx_in_k")
    mlflow.log_param("features", str(F_NAMES))
    _rid = _run.info.run_id
    log.info("MLflow run: %s", _rid)
_out = PATHS["metrics"] / "set-transformer-architecture.png"

tf.keras.utils.plot_model(
    model,
    to_file=str(_out),
    show_shapes=True,
    show_layer_names=True,
    dpi=150
)

log.info("Saved: %s", _out)

if MLFLOW_OK:
    mlflow.log_artifact(str(_out))
    _rid = None

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor=MONITOR, patience=12, min_delta=1e-4,
        restore_best_weights=True, mode="max", verbose=1),
    keras.callbacks.ModelCheckpoint(
        filepath=CKPT_PATH, monitor=MONITOR,
        save_best_only=True, mode="max", verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor=MONITOR, factor=0.5, patience=6,
        min_lr=1e-7, mode="max", verbose=1),
    keras.callbacks.TensorBoard(
        log_dir=str(PATHS["tb_logs"]),
        histogram_freq=0, write_graph=True, update_freq="epoch"),
    keras.callbacks.CSVLogger(
        str(PATHS["metrics"]/"training_log.csv"), append=False),
    MLflowEpochCB(),
]
log.info("Monitor='%s'  CKPT='%s'", MONITOR, CKPT_PATH)

🏃 View run 6g_predictive_exp4 at: http://127.0.0.1:5000/#/experiments/17/runs/655cdee406a34c739b645d92ad625553
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/17
13:06:28 │ INFO     │ MLflow run: 747fd12ccab8478985085311d8f6944a
13:06:28 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/6g_predictive/set-transformer-architecture.png
13:06:30 │ INFO     │ Monitor='val_cls_output_top1_acc'  CKPT='/home/wassimmchichi/Downloads/Handover_projects/models/best_6g_predictive.keras'


In [54]:
# ─── Training ────────────────────────────────────────────────────────────────

log.info("Training — %d train | %d val | batch=%d | max_ep=%d",
         len(y_tr), len(y_va), HP["BATCH_SIZE"], HP["EPOCHS"])

history = model.fit(
    ds_tr,
    validation_data = ds_va,
    epochs          = HP["EPOCHS"],
    callbacks       = callbacks,
    verbose         = 1,
    # class_weight argument has been intentionally removed
    # Weights are now handled automatically by ds_tr
)

best_ep   = int(np.argmax(history.history[MONITOR])) + 1
best_top1 = float(max(history.history[MONITOR]))
_mae_key  = "val_reg_output_rsrp_mae"
best_mae  = float(history.history[_mae_key][best_ep-1]) if _mae_key in history.history else -1

log.info("Done — best %s=%.4f @ ep %d  RSRP_MAE=%.4f",
         MONITOR, best_top1, best_ep, best_mae)

if MLFLOW_OK:
    mlflow.log_metrics({"best_val_top1.keras": best_top1,
                         "best_val_rsrp_mae.keras": best_mae,
                         "best_epoch.keras": float(best_ep)})

13:06:51 │ INFO     │ Training — 20370 train | 4365 val | batch=64 | max_ep=60
Epoch 1/60
318/319 [============================>.] - ETA: 0s - loss: 1.0336 - cls_output_loss: 0.3841 - reg_output_loss: 0.9278 - cls_output_top1_acc: 0.2881 - cls_output_top3_acc: 0.6040 - reg_output_rsrp_mae: 0.7118
Epoch 1: val_cls_output_top1_acc improved from -inf to 0.51029, saving model to /home/wassimmchichi/Downloads/Handover_projects/models/best_6g_predictive.keras
319/319 [==============================] - 34s 64ms/step - loss: 1.0333 - cls_output_loss: 0.3841 - reg_output_loss: 0.9275 - cls_output_top1_acc: 0.2882 - cls_output_top3_acc: 0.6041 - reg_output_rsrp_mae: 0.7116 - val_loss: 0.4829 - val_cls_output_loss: 0.2462 - val_reg_output_loss: 0.3382 - val_cls_output_top1_acc: 0.5103 - val_cls_output_top3_acc: 0.8025 - val_reg_output_rsrp_mae: 0.4524 - lr: 3.3552e-04
Epoch 2/60
318/319 [============================>.] - ETA: 0s - loss: 0.5938 - cls_output_loss: 0.2648 - reg_output_loss: 0.4700 -

In [55]:
_out = PATHS["metrics"] / "set-transformer.png"

tf.keras.utils.plot_model(
    model,
    to_file=str(_out),
    show_shapes=True,
    show_layer_names=True,
    dpi=150
)

log.info("Saved: %s", _out)

if MLFLOW_OK:
    mlflow.log_artifact(str(_out))

13:17:47 │ INFO     │ Saved: /home/wassimmchichi/Downloads/Handover_projects/metrics/6g_predictive/set-transformer.png


## Section 10 · Training Curves

In [56]:
# ─── Section 10 · Curves ─────────────────────────────────────────────────────

hist=history.history; ep=range(1,len(hist["loss"])+1)
fig,axes=plt.subplots(1,4,figsize=(20,4.5))
for ax,(tr_k,va_k,title,hi) in zip(axes,[
    ("loss","val_loss","Total Loss",False),
    ("cls_output_top1_acc","val_cls_output_top1_acc","Top-1 Acc (KPI)",True),
    ("cls_output_top3_acc","val_cls_output_top3_acc","Top-3 Accuracy",True),
    ("reg_output_rsrp_mae","val_reg_output_rsrp_mae","RSRP MAE (diag)",False),
]):
    if tr_k not in hist: continue
    ax.plot(ep,hist[tr_k],lw=2,label="train")
    ax.plot(ep,hist[va_k],lw=2,ls="--",label="val")
    fn=np.argmax if hi else np.argmin
    be=fn(hist[va_k])+1; bv=(max if hi else min)(hist[va_k])
    ax.axvline(be,color="red",ls=":",lw=1.2,alpha=0.8)
    ax.scatter([be],[bv],color="red",zorder=5,s=70,
               label=f"best@ep{be}({bv:.4f})")
    ax.set(title=title,xlabel="Epoch"); ax.legend(fontsize=7); ax.grid(alpha=0.4)
fig.suptitle("6G Predictive MTL — Experiment 4",fontsize=13,fontweight="bold")
plt.tight_layout()
_out=PATHS["metrics"]/"6g_training_curves.png"
plt.savefig(str(_out),dpi=150,bbox_inches="tight"); plt.close()
if MLFLOW_OK: mlflow.log_artifact(str(_out))

## Section 11 · Evaluation — Load `best_6g_predictive.keras`

In [57]:
# ─── Section 11 · Evaluation ─────────────────────────────────────────────────

model = keras.models.load_model(
    CKPT_PATH,
    custom_objects={"MaskedMultiHeadAttention": MaskedMultiHeadAttention,
                    "SetTransformerBlock"      : SetTransformerBlock,
                    "WarmUpCosineDecay"         : WarmUpCosineDecay, FOCAL_LOSS.__name__: FOCAL_LOSS},
    compile=False
)
log.info("Loaded: %s", CKPT_PATH)

preds     = model.predict(ds_te, verbose=1)
probs_te  = preds["cls_output"]         # (N, H, 10)
rsrp_pred = preds["reg_output"]         # (N, H)
y_pred_te = probs_te.argmax(axis=-1)

# Flatten sequences for global metric evaluation
y_te_flat = y_te.ravel()
y_pred_te_flat = y_pred_te.ravel()
probs_te_flat = probs_te.reshape(-1, probs_te.shape[-1])

# Classification
top1 = float((y_pred_te_flat == y_te_flat).mean())
top3 = float(top_k_accuracy_score(y_te_flat, probs_te_flat, k=3, labels=ALL_LABELS))
top5 = float(top_k_accuracy_score(y_te_flat, probs_te_flat, k=5, labels=ALL_LABELS))

# Regression (inverse-scale to dBm)
if 'scaler_r' not in globals(): scaler_r = _data["scalers"]["r"]
rsrp_pred_dbm = scaler_r.inverse_transform(rsrp_pred.reshape(-1,1)).reshape(rsrp_pred.shape)
rsrp_true_dbm = scaler_r.inverse_transform(r_te.reshape(-1,1)).reshape(r_te.shape)
rsrp_mae_dbm  = float(mean_absolute_error(rsrp_true_dbm.ravel(), rsrp_pred_dbm.ravel()))

log.info("Test → Top-1:%.4f  Top-3:%.4f  RSRP_MAE:%.2f dBm",
         top1, top3, rsrp_mae_dbm)
if MLFLOW_OK:
    mlflow.log_metrics({"test_top1":top1,"test_top3":top3,
                         "test_top5":top5,"test_rsrp_mae_dbm":rsrp_mae_dbm})

print("=" * 65)
print("  EXPERIMENT 4 — 6G PREDICTIVE TEST RESULTS")
print("=" * 65)
print(f"  Top-1 : {top1:.4f}   ({top1*100:.2f}%)")
print(f"  Top-3 : {top3:.4f}   ({top3*100:.2f}%)")
print(f"  Top-5 : {top5:.4f}   ({top5*100:.2f}%)")
print(f"  RSRP MAE: {rsrp_mae_dbm:.2f} dBm")
print()
print(classification_report(
    y_te_flat, y_pred_te_flat,
    target_names=[f"Cell{i}" for i in range(HP["MAX_CELLS"])],
    labels=ALL_LABELS,
    digits=4, zero_division=0,
))

13:17:55 │ INFO     │ Loaded: /home/wassimmchichi/Downloads/Handover_projects/models/best_6g_predictive.keras
69/69 [==============================] - 3s 20ms/step
13:17:58 │ INFO     │ Test → Top-1:0.5493  Top-3:0.8547  RSRP_MAE:5.23 dBm
  EXPERIMENT 4 — 6G PREDICTIVE TEST RESULTS
  Top-1 : 0.5493   (54.93%)
  Top-3 : 0.8547   (85.47%)
  Top-5 : 0.9470   (94.70%)
  RSRP MAE: 5.23 dBm

              precision    recall  f1-score   support

       Cell0     0.5664    0.5153    0.5396      2509
       Cell1     0.5342    0.5456    0.5398      2062
       Cell2     0.5731    0.5602    0.5666      2183
       Cell3     0.5480    0.5316    0.5397      2212
       Cell4     0.5531    0.5893    0.5706      2016
       Cell5     0.5489    0.5821    0.5650      2187
       Cell6     0.5653    0.5399    0.5523      2132
       Cell7     0.5241    0.5360    0.5300      2192
       Cell8     0.5644    0.5669    0.5657      2272
       Cell9     0.5153    0.5325    0.5238      2060

    accuracy   

## Section 12 · Confusion Matrix & Experiment Comparison

In [58]:
# ─── Section 12 · Confusion matrix ───────────────────────────────────────────

C=HP["MAX_CELLS"]
cm=confusion_matrix(y_te.flatten(), y_pred_te.flatten(),labels=list(range(C)))
cm_norm=cm.astype(float)/(cm.sum(axis=1,keepdims=True)+1e-9)
cl=[f"C{i}" for i in range(C)]

fig,axes=plt.subplots(1,2,figsize=(16,6))
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",
            xticklabels=[f"P-{l}" for l in cl],
            yticklabels=[f"T-{l}" for l in cl],
            linewidths=0.5,ax=axes[0],annot_kws={"size":9})
axes[0].set(title="Confusion Matrix — Counts",ylabel="Actual",xlabel="Predicted")
sns.heatmap(cm_norm,annot=True,fmt=".2f",cmap="YlGn",
            xticklabels=[f"P-{l}" for l in cl],
            yticklabels=[f"T-{l}" for l in cl],
            linewidths=0.5,ax=axes[1],vmin=0,vmax=1,annot_kws={"size":9})
axes[1].set(title="Normalised Recall",ylabel="Actual",xlabel="Predicted")
plt.tight_layout()
_out=PATHS["metrics"]/"confusion_matrix.png"
plt.savefig(str(_out),dpi=150,bbox_inches="tight"); plt.close()
if MLFLOW_OK: mlflow.log_artifact(str(_out))

per_recall=cm_norm.diagonal()
fig,ax=plt.subplots(figsize=(9,3.5))
bars=ax.bar(range(C),per_recall,
            color=["#2196F3" if v>=0.5 else "#F44336" for v in per_recall],
            edgecolor="white")
ax.axhline(top1,color="black",ls="--",lw=1.2,label=f"Overall Top-1 ({top1:.3f})")
for b,v in zip(bars,per_recall):
    ax.text(b.get_x()+b.get_width()/2,v+0.015,f"{v:.2f}",
            ha="center",va="bottom",fontsize=8)
ax.set(xticks=range(C),xticklabels=[f"C{i}" for i in range(C)],
       ylabel="Recall",ylim=(0,1.18),title="Per-Cell Recall — 6G Predictive")
ax.legend(); ax.grid(axis="y",alpha=0.4)
plt.xticks(rotation=30,ha="right"); plt.tight_layout()
_out=PATHS["metrics"]/"per_cell_recall.png"
plt.savefig(str(_out),dpi=150,bbox_inches="tight"); plt.close()
if MLFLOW_OK: mlflow.log_artifact(str(_out))

# Experiment progression comparison
EXPS = {
    "Exp1\nDeepSet": {"top1":0.53,"top3":0.78},
    "Exp2\nSetTrans": {"top1":0.57,"top3":0.82},
    "Exp3\nMTL Δ-feat": {"top1":0.57,"top3":0.83},
    "Exp4\n6G Predict": {"top1":top1,"top3":top3},
}
_names=list(EXPS.keys()); _t1=[v["top1"] for v in EXPS.values()]
_t3=[v["top3"] for v in EXPS.values()]
_colors=["#B0BEC5","#78909C","#546E7A","#DD8452"]
x=np.arange(len(_names)); w=0.35
fig,ax=plt.subplots(figsize=(11,5))
b1=ax.bar(x-w/2,_t1,w,label="Top-1",color=_colors,edgecolor="white")
b3=ax.bar(x+w/2,_t3,w,label="Top-3",color=_colors,alpha=0.45,
          hatch="///",edgecolor="white")
ax.axhline(0.57,color="red",ls="--",lw=1.2,label="57% ceiling")
for b,v in zip(b1,_t1): ax.text(b.get_x()+b.get_width()/2,v+0.005,
    f"{v:.3f}",ha="center",fontsize=9,fontweight="bold")
ax.set(xticks=x,xticklabels=_names,ylabel="Accuracy",ylim=(0,1.1),
       title="Experiment Progression — Handover Cell Selection")
ax.legend(fontsize=10); ax.grid(axis="y",alpha=0.4)
plt.tight_layout()
_out=PATHS["metrics"]/"exp4_comparison.png"
plt.savefig(str(_out),dpi=150,bbox_inches="tight"); plt.close()
log.info("Saved comparison: %s", _out)
if MLFLOW_OK: mlflow.log_artifact(str(_out))

13:18:08 │ INFO     │ Saved comparison: /home/wassimmchichi/Downloads/Handover_projects/metrics/6g_predictive/exp4_comparison.png


## Section 13 · Handover-Only Evaluation

### Why evaluate on handover sequences separately?

The overall test set is dominated by **no-handover** samples (optimal cell = serving cell, i.e. `y_bin=0`).  
A model can achieve >50% accuracy by simply predicting "stay on serving cell" for every sample.

This section filters the test set to only those sequences where the ground truth
label indicates an **actual handover event** (`y_bin == 1` — the optimal cell is a
different cell than the current serving cell).  
This gives us the **handover detection accuracy** — the metric that matters most
for real-world deployment, where failing to predict a necessary handover causes
radio link failure (RLF).

| Metric | What it measures |
|--------|------------------|
| `ho_top1` | Exact match on the correct target cell during handover |
| `ho_top3` | Target cell is in the model's top-3 predictions |
| `ho_top5` | Target cell is in the model's top-5 predictions |
| `ho_rsrp_mae` | RSRP prediction error during handover events |

In [59]:
# ─── Section 13 · Handover-Only Evaluation ───────────────────────────────────
#
# Filter test set to sequences where a handover actually occurs:
#   y_bin_te == 1  →  the optimal cell is NOT the serving cell
#
# This isolates model performance on the decision that matters most:
# "which neighbour cell should the UE hand over to?"

y_bin_te_flat = y_bin_te.ravel()
ho_mask = (y_bin_te_flat == 1)  # True where optimal cell != serving cell
n_ho    = int(ho_mask.sum())
n_total = len(y_te_flat)

log.info("Handover-only subset: %d / %d samples (%.1f%%)",
         n_ho, n_total, 100.0 * n_ho / n_total)

if n_ho == 0:
    log.warning("No handover events in test set — skipping HO-only evaluation.")
else:
    # ── Slice predictions and ground truth ────────────────────────────────────
    y_te_ho      = y_te_flat[ho_mask]
    y_pred_te_ho = y_pred_te_flat[ho_mask]
    probs_te_ho  = probs_te_flat[ho_mask]

    # ── Classification metrics (handover-only) ───────────────────────────────
    ho_top1 = float((y_pred_te_ho == y_te_ho).mean())
    ho_top3 = float(top_k_accuracy_score(
        y_te_ho, probs_te_ho, k=3, labels=ALL_LABELS))
    ho_top5 = float(top_k_accuracy_score(
        y_te_ho, probs_te_ho, k=5, labels=ALL_LABELS))

    # ── Regression metrics (handover-only, inverse-scale to dBm) ─────────────
    if 'scaler_r' not in globals(): scaler_r = _data["scalers"]["r"]
    ho_rsrp_pred = scaler_r.inverse_transform(
        rsrp_pred.ravel()[ho_mask].reshape(-1, 1)).ravel()
    ho_rsrp_true = scaler_r.inverse_transform(
        r_te.ravel()[ho_mask].reshape(-1, 1)).ravel()
    ho_rsrp_mae  = float(mean_absolute_error(ho_rsrp_true, ho_rsrp_pred))

    # ── Log to MLflow ────────────────────────────────────────────────────────
    if MLFLOW_OK:
        mlflow.log_metrics({
            "ho_only_top1": ho_top1, "ho_only_top3": ho_top3,
            "ho_only_top5": ho_top5, "ho_only_rsrp_mae_dbm": ho_rsrp_mae,
            "ho_only_n_samples": float(n_ho),
        })

    # ── Print results ────────────────────────────────────────────────────────
    print("=" * 65)
    print("  HANDOVER-ONLY EVALUATION (y_bin == 1)")
    print("=" * 65)
    print(f"  Samples   : {n_ho} / {n_total}  ({100*n_ho/n_total:.1f}% of test set)")
    print(f"  Top-1     : {ho_top1:.4f}   ({ho_top1*100:.2f}%)")
    print(f"  Top-3     : {ho_top3:.4f}   ({ho_top3*100:.2f}%)")
    print(f"  Top-5     : {ho_top5:.4f}   ({ho_top5*100:.2f}%)")
    print(f"  RSRP MAE  : {ho_rsrp_mae:.2f} dBm")
    print()

    # ── Comparison table: overall vs handover-only ───────────────────────────
    print(f"  {'Metric':<14} {'Overall':>10} {'HO-Only':>10} {'Δ':>8}")
    print(f"  {'─'*14} {'─'*10} {'─'*10} {'─'*8}")
    for lbl, ov, ho in [("Top-1", top1, ho_top1),
                         ("Top-3", top3, ho_top3),
                         ("Top-5", top5, ho_top5)]:
        d = ho - ov
        print(f"  {lbl:<14} {ov:>10.4f} {ho:>10.4f} {d:>+8.4f}")
    print(f"  {'RSRP MAE':<14} {rsrp_mae_dbm:>8.2f} dB {ho_rsrp_mae:>8.2f} dB"
          f" {ho_rsrp_mae-rsrp_mae_dbm:>+8.2f}")
    print()

    # ── Per-cell classification report (handover-only) ───────────────────────
    # Since neighbors are shuffled, the target cell during a handover can be at any index.
    print(classification_report(
        y_te_ho, y_pred_te_ho,
        labels=ALL_LABELS,
        target_names=[f"Cell{i}" for i in ALL_LABELS],
        digits=4, zero_division=0,
    ))

    # ── Confusion matrix (handover-only) ─────────────────────────────────────
    C = HP["MAX_CELLS"]
    cm_ho = confusion_matrix(y_te_ho.flatten(), y_pred_te_ho.flatten(), labels=list(range(C)))
    cm_ho_norm = cm_ho.astype(float) / (cm_ho.sum(axis=1, keepdims=True) + 1e-9)
    cl = [f"C{i}" for i in range(C)]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(cm_ho, annot=True, fmt="d", cmap="Blues",
                xticklabels=[f"P-{l}" for l in cl],
                yticklabels=[f"T-{l}" for l in cl],
                linewidths=0.5, ax=axes[0], annot_kws={"size": 9})
    axes[0].set(title="HO-Only Confusion — Counts",
                ylabel="Actual", xlabel="Predicted")

    sns.heatmap(cm_ho_norm, annot=True, fmt=".2f", cmap="YlGn",
                xticklabels=[f"P-{l}" for l in cl],
                yticklabels=[f"T-{l}" for l in cl],
                linewidths=0.5, ax=axes[1], vmin=0, vmax=1,
                annot_kws={"size": 9})
    axes[1].set(title="HO-Only Normalised — Recall per Row",
                ylabel="Actual", xlabel="Predicted")

    fig.suptitle("Handover-Only Evaluation (y_bin == 1)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    _out = PATHS["metrics"] / "ho_only_confusion_matrix.png"
    plt.savefig(str(_out), dpi=150, bbox_inches="tight"); plt.close()
    log.info("Saved: %s", _out)
    if MLFLOW_OK: mlflow.log_artifact(str(_out))


13:18:09 │ INFO     │ Handover-only subset: 17802 / 21825 samples (81.6%)
  HANDOVER-ONLY EVALUATION (y_bin == 1)
  Samples   : 17802 / 21825  (81.6% of test set)
  Top-1     : 0.4939   (49.39%)
  Top-3     : 0.8334   (83.34%)
  Top-5     : 0.9383   (93.83%)
  RSRP MAE  : 5.36 dBm

  Metric            Overall    HO-Only        Δ
  ────────────── ────────── ────────── ────────
  Top-1              0.5493     0.4939  -0.0554
  Top-3              0.8547     0.8334  -0.0213
  Top-5              0.9470     0.9383  -0.0087
  RSRP MAE           5.23 dB     5.36 dB    +0.13

              precision    recall  f1-score   support

       Cell0     0.5169    0.4538    0.4833      2089
       Cell1     0.4754    0.4833    0.4793      1678
       Cell2     0.5090    0.4957    0.5023      1761
       Cell3     0.4768    0.4663    0.4715      1765
       Cell4     0.5070    0.5431    0.5244      1659
       Cell5     0.4997    0.5400    0.5191      1813
       Cell6     0.5133    0.4893    0.5010    

## Section 14 · Save Final Model, Metadata & Close MLflow

In [60]:
# ─── Section 14 · Persist ────────────────────────────────────────────────────

model.save(FINAL_PATH)
log.info("Final model: %s", FINAL_PATH)

meta = {
    "experiment"      : "Experiment 4 — 6G Predictive Optimal Cell Selection",
    "created"         : datetime.datetime.now().isoformat(),
    "notebook"        : "notebooks/modeling/04_6g_predictive.ipynb",
    "target_column"   : "optimal_cell_idx_in_k",
    "best_checkpoint.keras" : CKPT_PATH,
    "final_model"     : FINAL_PATH,
    "test_top1_acc"   : round(top1, 4),
    "test_top3_acc"   : round(top3, 4),
    "test_top5_acc"   : round(top5, 4),
    "test_rsrp_mae_dbm": round(rsrp_mae_dbm, 3),
    "best_epoch.keras"      : best_ep,
    "feature_names"   : F_NAMES,
    "hyperparams"     : HP,
    "improvement_vs_57": round(top1 - 0.57, 4),
    "paths"           : {k: str(v) for k, v in PATHS.items()},
}
_mout = PATHS["metrics"] / "6g_predictive_metadata.json"
json.dump(meta, open(str(_mout),"w"), indent=2)

if MLFLOW_OK:
    for p in PATHS["metrics"].glob("*.png"):
        mlflow.log_artifact(str(p))
    mlflow.log_artifact(str(_mout))
    mlflow.tensorflow.log_model(
        model, artifact_path="6g_predictive_keras",
        registered_model_name="handover_6g_predictive")
    mlflow.end_run()
    log.info("MLflow run closed.")

print()
print("=" * 65)
print("  ARTEFACT INVENTORY")
print("=" * 65)
for lbl,d in [("models/",PATHS["models"]),("metrics/",PATHS["metrics"])]:
    print(f"\n  {lbl}")
    for p in sorted(Path(d).iterdir()):
        if p.is_file():
            print(f"    {p.name:<44s}{p.stat().st_size/1024:>7.1f} KB")
print()
print(f"  Top-1 : {top1:.4f}  (prev 0.5700  Δ={top1-0.57:+.4f})")
print(f"  Top-3 : {top3:.4f}")
print(f"  RSRP MAE: {rsrp_mae_dbm:.2f} dBm")
print()
print("  TensorBoard: tensorboard --logdir ../../tb_logs/")
print("  MLflow:      mlflow ui --backend-store-uri file://$(pwd)/../../mlflow/mlruns")

13:18:13 │ INFO     │ Final model: /home/wassimmchichi/Downloads/Handover_projects/models/6g_predictive_final.keras


2026/05/30 13:18:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 13:18:27 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


13:18:27 │ WARNING  │ Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
INFO:tensorflow:Assets written to: /tmp/tmp8z1s0v4b/model/data/model/assets
13:18:35 │ INFO     │ Assets written to: /tmp/tmp8z1s0v4b/model/data/model/assets


Registered model 'handover_6g_predictive' already exists. Creating a new version of this model...
2026/05/30 13:19:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: handover_6g_predictive, version 2
Created version '2' of model 'handover_6g_predictive'.


🏃 View run 6G_predictive_ at: http://127.0.0.1:5000/#/experiments/17/runs/747fd12ccab8478985085311d8f6944a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/17
13:19:17 │ INFO     │ MLflow run closed.

  ARTEFACT INVENTORY

  models/
    6g_predictive_final.keras                    1476.9 KB
    best_6g_predictive.keras                     6888.3 KB
    best_honet_final.keras                       3281.3 KB
    best_honet_p1.keras                          1545.1 KB
    best_honet_p2.keras                          2516.3 KB
    best_mh_transformer.keras                    3407.2 KB
    best_mtl_transformer.keras                   2096.9 KB
    best_set_transformer.keras                   2650.9 KB
    best_st_deepset.keras                        9277.2 KB
    best_strategic_deepset.keras                 2404.2 KB
    best_temporal_deepset.keras                   764.1 KB
    cell_scaler.pkl                                 0.6 KB
    mtl_transformer_final.keras                   